# RQ1: Variation of the Betula pollen season in time and space

This notebook describes how the onset, end, length and intensity (SPIn) of the Betula pollen season vary between years and between Helmond and Leiden, and uses KNMI temperatures and city sensors in Enschede to estimate at what spatial scale onset differences are measurable.
Onset and end are the days on which 5 % and 95 % of the seasonal sum are reached, SPIn is the sum from onset to end, and GDD uses T_base = 3.5 °C from 1 March.

**Data (not included in this repository):** pollen counts of Helmond (Elkerliek, one sheet per year) and Leiden (LUMC), KNMI daily data (downloaded automatically if missing) and the daily values of the Enschede sensor network. The 1 km analysis is in the `rq1_hrm_*` notebooks.

## Step 1: Settings
Set the analysis settings, file paths, locations and KNMI stations.

In [ ]:
import os, io, re, glob, math, zipfile, datetime, calendar, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, FixedFormatter, NullFormatter
from IPython.display import display
warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

# analysis
YEARS = list(range(2016, 2027))   # common period of Helmond and Leiden
ONSET_PCT = 0.05
END_PCT = 0.95
WINDOW = ((1, 1), (9, 30))        # seasonal sum over the counting period, 1 January to 30 September
CORE_SEASON = ((3, 1), (5, 15))   # missing days in this period are reported
MAX_MISSING_CORE = 5              # more missing days in the core period = year not used
T_BASE = 3.5                      # °C
D0 = (3, 1)                       # start of GDD accumulation: 1 March
FIG_YEAR = 2026

# counting noise (Step 5)
N_MC = 1000
NOISE_SIGMA = 0.17                # daily log-normal noise, about 16 % difference between two Hirst traps (Adamov et al., 2024)
SEED = 42

# files
DATA_DIR = 'data'
OUT_DIR = 'output_rq1'
POLLEN_HELMOND = os.path.join(DATA_DIR, 'pollen_helmond_all_years.xlsx')
POLLEN_LEIDEN = os.path.join(DATA_DIR, 'pollen_leiden.xlsx')
KNMI_DIR = os.path.join(DATA_DIR, 'knmi')   # KNMI daily files (etmgeg), missing stations are downloaded here
DOWNLOAD_KNMI = True
SENSOR_CSV = os.path.join(DATA_DIR, 'enschede_envmon_daily.csv')
os.makedirs(OUT_DIR, exist_ok=True)

# locations and stations
LOCATIONS = {                              # name: (lat, lon)
    'Helmond': (51.482303, 5.642436),      # pollen station Elkerliek hospital
    'Leiden': (52.166031, 4.478504),       # pollen station LUMC
    'Enschede': (52.2215, 6.8937),         # city centre, no pollen counts
}
POLLEN_SITES = ['Helmond', 'Leiden']
STATIONS = {                               # KNMI number: (name, lat, lon)
    215: ('Voorschoten', 52.141, 4.437),
    235: ('De Kooy', 52.928, 4.781),
    260: ('De Bilt', 52.100, 5.180),
    286: ('Nieuw Beerta', 53.196, 7.150),
    290: ('Twenthe', 52.274, 6.891),
    310: ('Vlissingen', 51.442, 3.596),
    370: ('Eindhoven', 51.451, 5.377),
}
NATIONAL_STATION = 260
REGIONAL_STATION = {'Helmond': 370, 'Leiden': 215, 'Enschede': 290}

# figures
COLORS = {'Helmond': '#E64A19', 'Leiden': '#388E3C', 'Enschede': '#1976D2'}
MARKERS = {'Helmond': 'o', 'Leiden': 's', 'Enschede': '^'}
LAYER_STYLE = {'National': ('#2a78d6', 'D'), 'Regional': ('#4a3aa7', 'o')}
FIG_DPI = 200
OUT_XLSX = os.path.join(OUT_DIR, 'rq1_results.xlsx')

def km(a, b):
    # distance in km between two (lat, lon) points
    la1, lo1, la2, lo2 = map(math.radians, (a[0], a[1], b[0], b[1]))
    h = math.sin((la2 - la1) / 2) ** 2 + math.cos(la1) * math.cos(la2) * math.sin((lo2 - lo1) / 2) ** 2
    return 2 * 6371.0 * math.asin(math.sqrt(h))

def fmt_date(d, year=False):
    if d is None or pd.isna(d):
        return ''
    return pd.Timestamp(d).strftime('%d %b %Y' if year else '%d %b')

def fmt_doy(doy):
    # day of year as date, e.g. 82 -> '23 Mar' (non-leap year)
    if doy is None or pd.isna(doy):
        return ''
    return (pd.Timestamp(2001, 1, 1) + pd.Timedelta(days=int(round(float(doy))) - 1)).strftime('%d %b')

def same_day_2001(d):
    # same day and month in 2001, so seasons of different years share one axis
    d = pd.Timestamp(d)
    return pd.Timestamp(2001, d.month, min(d.day, 28) if d.month == 2 else d.day)

def binom_sf(k, n, p):
    # P(X >= k) for X ~ Binomial(n, p)
    return float(sum(math.comb(n, i) * p ** i * (1 - p) ** (n - i) for i in range(k, n + 1)))

def sign_test(diffs):
    # two-sided sign test, zeros are ignored
    d = [x for x in diffs if pd.notna(x) and x != 0]
    n, k = len(d), sum(1 for x in d if x > 0)
    return np.nan if n == 0 else min(1.0, 2 * binom_sf(max(k, n - k), n, 0.5))

def tidy(ax, grid='y'):
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)
    if grid:
        ax.grid(axis=grid, color='#d9d9d9', lw=0.6)
        ax.set_axisbelow(True)

def savefig(fig, name):
    fig.savefig(os.path.join(OUT_DIR, name), dpi=FIG_DPI, bbox_inches='tight')

results = {}   # all tables for the export (Step 9)
S = {}         # key numbers for the summary (Step 8)
print(f"Distance Helmond to Leiden: {km(LOCATIONS['Helmond'], LOCATIONS['Leiden']):.0f} km")

## Step 2: Pollen counts
Read the daily Betula counts and check for missing days in the core period of 1 March to 15 May.

In [ ]:
def _is_date(v):
    return isinstance(v, (pd.Timestamp, datetime.datetime)) and pd.notna(v)

def read_helmond(path):
    # one sheet per year with dates in the columns and taxa in the rows,
    # day and month come from the header and the year from the sheet name,
    # an empty cell is 0 grains and text is a missing value
    log, frames = [], []
    xl = pd.ExcelFile(path)
    for sn in xl.sheet_names:
        if not sn.strip().isdigit():
            continue
        year = int(sn.strip())
        raw = xl.parse(sn, header=None)
        n_dates = [sum(_is_date(v) for v in raw.iloc[i]) for i in range(min(len(raw), 6))]
        drow = int(np.argmax(n_dates))
        brow = next((i for i in range(len(raw)) if str(raw.iat[i, 0]).strip().lower() == 'betula'), None)
        if n_dates[drow] < 100 or brow is None:
            log.append(f'Helmond {year}: date or Betula row not found, year skipped')
            continue
        dates, vals, seen_jan = [], [], False
        for j, v in enumerate(raw.iloc[drow]):
            if not _is_date(v):
                continue
            seen_jan = seen_jan or v.month == 1
            if v.month == 12 and not seen_jan:   # December columns before January belong to the previous year
                continue
            if (v.month, v.day) == (2, 29) and not calendar.isleap(year):
                log.append(f'Helmond {year}: 29 February column skipped')
                continue
            cell = raw.iat[brow, j]
            if isinstance(cell, str):
                val = pd.to_numeric(cell.strip().replace(',', '.'), errors='coerce')
            elif pd.isna(cell):
                val = 0.0
            else:
                val = float(cell)
            dates.append(pd.Timestamp(year, v.month, v.day))
            vals.append(val)
        s = pd.Series(vals, index=pd.DatetimeIndex(dates))
        rest = s.iloc[:-1].sum()
        if len(s) > 1 and rest > 0 and abs(s.iloc[-1] - rest) <= max(2.0, 0.02 * rest):
            log.append(f'Helmond {year}: last column ({s.index[-1]:%d-%m}) is the annual total, not counted')
            s = s.iloc[:-1]
        if s.index.duplicated().any():
            dup = s.index[s.index.duplicated()]
            log.append(f'Helmond {year}: {len(dup)} duplicate date(s) from {dup[0]:%d-%m}, first value kept')
            s = s[~s.index.duplicated()]
        frames.append(pd.DataFrame({'location': 'Helmond', 'date': s.index, 'betula': s.values}))
    return pd.concat(frames, ignore_index=True).sort_values('date'), log

def read_leiden(path):
    # one row per day with the columns date and Betula, an empty cell is a missing value
    xl = pd.ExcelFile(path)
    for sn in xl.sheet_names:
        d = xl.parse(sn)
        cols = {str(c).strip().lower(): c for c in d.columns}
        if 'date' in cols and 'betula' in cols:
            out = pd.DataFrame({'location': 'Leiden',
                                'date': pd.to_datetime(d[cols['date']], errors='coerce'),
                                'betula': pd.to_numeric(d[cols['betula']], errors='coerce')})
            out = out.dropna(subset=['date'])
            return out[~out['date'].duplicated()].sort_values('date')
    raise ValueError(f'No sheet with the columns date and Betula in {path}')

ph, load_log = read_helmond(POLLEN_HELMOND)
pl = read_leiden(POLLEN_LEIDEN)
pollen = pd.concat([ph, pl], ignore_index=True)
pollen['year'] = pollen['date'].dt.year

def qc_row(loc, year, g):
    a, b = CORE_SEASON
    core = pd.date_range(pd.Timestamp(year, *a), pd.Timestamp(year, *b), freq='D')
    s = g.set_index('date')['betula'].reindex(core)
    miss = s[s.isna()].index
    return {'location': loc, 'year': year, 'first_day': g['date'].min().date(), 'last_day': g['date'].max().date(),
            'n_days': len(g), 'missing_core_days': len(miss),
            'missing_core_dates': ', '.join(d.strftime('%d-%m') for d in miss[:8]) + (' ...' if len(miss) > 8 else ''),
            'usable': len(miss) <= MAX_MISSING_CORE}

qc = pd.DataFrame([qc_row(loc, yr, g) for (loc, yr), g in pollen.groupby(['location', 'year'])])
results['data_check'] = qc
print('\n'.join(load_log) if load_log else 'No reading issues')
for loc, g in qc.groupby('location'):
    print(f"{loc}: {g.year.min()} to {g.year.max()}, {g.usable.sum()} usable years")
display(qc[qc.missing_core_days > 0])

## Step 3: Seasonal indicators
Compute the onset, end, length, SPIn and peak per station and year.

In [ ]:
def season_indicators(s, year):
    a, b = WINDOW
    s = s[(s.index >= pd.Timestamp(year, *a)) & (s.index <= pd.Timestamp(year, *b))].sort_index()
    v = s.fillna(0.0)   # a missing day counts as 0
    total = float(v.sum())
    if total <= 0:
        return None
    c = (v.cumsum() / total).values
    onset = v.index[int(np.argmax(c >= ONSET_PCT))]
    end = v.index[int(np.argmax(c >= END_PCT))]
    peak_day = v.idxmax()
    return {'onset': onset, 'end': end, 'length': (end - onset).days + 1,
            'SPIn': float(v.loc[onset:end].sum()), 'APIn': total,
            'peak_date': peak_day, 'peak': float(v.max()),
            'onset_doy': onset.dayofyear, 'end_doy': end.dayofyear, 'peak_doy': peak_day.dayofyear}

ok = set(zip(qc.loc[qc.usable, 'location'], qc.loc[qc.usable, 'year']))
rows = []
for (loc, yr), g in pollen.groupby(['location', 'year']):
    if (loc, yr) in ok:
        r = season_indicators(g.set_index('date')['betula'], yr)
        if r is not None:
            rows.append({'location': loc, 'year': yr, **r})
ind = pd.DataFrame(rows).sort_values(['location', 'year']).reset_index(drop=True)
ind_main = ind[ind['year'].isin(YEARS)].reset_index(drop=True)   # 2016 to 2026, both stations
results['indicators_all_years'] = ind

def table1(d):
    out = {}
    for loc in POLLEN_SITES:
        x = d[d.location == loc].set_index('year')
        out[(loc, 'Onset')] = x['onset'].map(fmt_date)
        out[(loc, 'End')] = x['end'].map(fmt_date)
        out[(loc, 'Length (d)')] = x['length']
        out[(loc, 'SPIn')] = x['SPIn'].round(0).astype(int)
        out[(loc, 'Peak (date, count)')] = pd.Series([f'{fmt_date(p)} ({v:.0f})' for p, v in zip(x['peak_date'], x['peak'])],
                                                     index=x.index)
    t = pd.DataFrame(out)
    t.index.name = 'Year'
    return t

tab1 = table1(ind_main)
results['table1_indicators'] = tab1
display(tab1)

# daily counts in FIG_YEAR with onset and end
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True, sharey=True)
ymax = pollen[(pollen.year == FIG_YEAR) & (pollen.date.dt.month.isin([3, 4, 5]))].betula.max()
axes[0].set_ylim(0, 1.2 * ymax)
for ax, loc in zip(axes, POLLEN_SITES):
    s = pollen[(pollen.location == loc) & (pollen.year == FIG_YEAR)].set_index('date')['betula']
    s = s.loc[f'{FIG_YEAR}-03-01':f'{FIG_YEAR}-05-31']
    r = ind[(ind.location == loc) & (ind.year == FIG_YEAR)]
    if len(r):
        r = r.iloc[0]
        ax.axvspan(r.onset, r.end, color=COLORS[loc], alpha=0.08, lw=0)
        for d, lab in [(r.onset, 'onset'), (r.end, 'end')]:
            ax.axvline(d, color='#444444', lw=1, ls='--')
            ax.text(d, 0.98, f' {lab} {fmt_date(d)}', transform=ax.get_xaxis_transform(), va='top', fontsize=9, color='#444444')
        ax.text(0.995, 0.70, f'SPIn = {r.SPIn:,.0f}\nlength = {r.length} days', transform=ax.transAxes,
                ha='right', va='top', fontsize=9, color='#222222')
    ax.bar(s.index, s.values, width=0.85, color=COLORS[loc])
    ax.set_title(loc, loc='left', fontsize=11, fontweight='bold')
    ax.set_ylabel('Betula (grains m$^{-3}$)')
    tidy(ax)
axes[-1].xaxis.set_major_locator(mdates.DayLocator(bymonthday=[1, 15]))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
fig.suptitle(f'Daily Betula pollen concentration in {FIG_YEAR}; shaded = season (5 %–95 %)', x=0.01, ha='left', fontsize=11)
fig.tight_layout()
savefig(fig, f'rq1_season_{FIG_YEAR}.png')
plt.show()

# season (onset to end) per year
yrs = sorted(ind_main.year.unique())
fig, ax = plt.subplots(figsize=(10, 0.45 * len(yrs) + 1.8))
for i, yr in enumerate(yrs):
    for k, loc in enumerate(POLLEN_SITES):
        r = ind_main[(ind_main.location == loc) & (ind_main.year == yr)]
        if r.empty:
            continue
        r = r.iloc[0]
        y = i - 0.17 if k == 0 else i + 0.17
        ax.plot([same_day_2001(r.onset), same_day_2001(r.end)], [y, y], color=COLORS[loc], lw=5, solid_capstyle='butt')
        ax.plot(same_day_2001(r.peak_date), y, marker='D', ms=5.5, color='white', mec='#222222', mew=1.1, zorder=3)
ax.set_yticks(range(len(yrs)))
ax.set_yticklabels(yrs)
ax.invert_yaxis()
ax.xaxis.set_major_locator(mdates.DayLocator(bymonthday=[1, 15]))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
tidy(ax, grid='x')
handles = [Line2D([], [], color=COLORS[l], lw=5, label=f'{l}: onset → end') for l in POLLEN_SITES]
handles.append(Line2D([], [], marker='D', ls='', color='white', mec='#222222', label='Peak day'))
ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.06), ncol=3, frameon=False)
ax.set_title('Betula pollen season (5 %–95 %) per year, Helmond and Leiden', loc='left', fontsize=11)
fig.tight_layout()
savefig(fig, 'rq1_season_timeline.png')
plt.show()

## Step 4: Variation between years
Summarise the variation between years, test for a biennial cycle in the SPIn and for a trend in the long Helmond series.

In [ ]:
def time_summary(d):
    rows = []
    for loc, g in d.groupby('location'):
        e, l = g.loc[g.onset_doy.idxmin()], g.loc[g.onset_doy.idxmax()]
        rows.append({'Location': loc, 'Years': f'{g.year.min()}–{g.year.max()} (n={len(g)})',
                     'Onset mean': fmt_doy(g.onset_doy.mean()), 'Onset SD (d)': round(g.onset_doy.std(), 1),
                     'Earliest onset': fmt_date(e.onset, True), 'Latest onset': fmt_date(l.onset, True),
                     'Onset range (d)': int(g.onset_doy.max() - g.onset_doy.min()),
                     'End mean': fmt_doy(g.end_doy.mean()), 'End SD (d)': round(g.end_doy.std(), 1),
                     'Length mean (d)': round(g.length.mean(), 1), 'Length SD (d)': round(g.length.std(), 1),
                     'SPIn mean': int(round(g.SPIn.mean())), 'SPIn min': int(round(g.SPIn.min())),
                     'SPIn max': int(round(g.SPIn.max())), 'SPIn CV (%)': int(round(100 * g.SPIn.std() / g.SPIn.mean()))})
    return pd.DataFrame(rows)

def biennial(g, col='SPIn'):
    # with a biennial cycle the change from the previous year alternates in sign (expected 2/3 by chance)
    s = g.set_index('year')[col].sort_index()
    s = s.reindex(range(int(s.index.min()), int(s.index.max()) + 1))
    x = np.log(s)
    ch = x.diff()
    pairs = [(ch[t - 1], ch[t]) for t in ch.index if (t - 1) in ch.index and pd.notna(ch[t]) and pd.notna(ch[t - 1])]
    n = len(pairs)
    k = sum(1 for a, b in pairs if np.sign(a) != np.sign(b))
    return {'Years': f'{int(g.year.min())}–{int(g.year.max())}', 'Year pairs': n, 'Alternations': k,
            'Share alternating': round(k / n, 2) if n else np.nan, 'Expected if random': 0.67,
            'p (more than random)': round(binom_sf(k, n, 2 / 3), 3) if n else np.nan,
            'Lag-1 autocorrelation log(SPIn)': round(x.corr(x.shift(1)), 2)}

def trend(g, col, log=False):
    g = g.dropna(subset=[col])
    x = g['year'].values.astype(float)
    y = np.log(g[col].values.astype(float)) if log else g[col].values.astype(float)
    b = np.polyfit(x, y, 1)[0]
    try:
        from scipy import stats
        p = stats.linregress(x, y).pvalue
    except Exception:
        p = np.nan
    return b, p

ts_main = time_summary(ind_main)
helmond_long = ind[ind.location == 'Helmond']
ts_long = time_summary(helmond_long).assign(Location='Helmond (long series)')
time_tab = pd.concat([ts_main, ts_long], ignore_index=True)
results['time_summary'] = time_tab
display(time_tab)

bien = {'Helmond 2016–2026': biennial(ind_main[ind_main.location == 'Helmond']),
        'Leiden 2016–2026': biennial(ind_main[ind_main.location == 'Leiden']),
        'Helmond long series': biennial(helmond_long)}
bien_tab = pd.DataFrame(bien).T
results['biennial_cycle'] = bien_tab.reset_index().rename(columns={'index': 'Series'})
display(bien_tab)

w = ind_main.pivot(index='year', columns='location', values=['onset_doy', 'SPIn'])
S['r_onset'] = w['onset_doy']['Helmond'].corr(w['onset_doy']['Leiden'])
S['r_spin'] = np.log(w['SPIn']['Helmond']).corr(np.log(w['SPIn']['Leiden']))
print(f"Correlation between stations (2016 to 2026): onset r = {S['r_onset']:.2f}, log(SPIn) r = {S['r_spin']:.2f}")

b_on, p_on = trend(helmond_long, 'onset_doy')
b_sp, p_sp = trend(helmond_long, 'SPIn', log=True)
S['trend_onset'], S['trend_onset_p'] = 10 * b_on, p_on
S['trend_spin'], S['trend_spin_p'] = 100 * (np.exp(10 * b_sp) - 1), p_sp
S['helmond_long_years'] = f'{helmond_long.year.min()}–{helmond_long.year.max()} (n={len(helmond_long)})'
print(f"Trend Helmond {S['helmond_long_years']}: onset {S['trend_onset']:+.1f} days per 10 years (p = {p_on:.2f}), "
      f"SPIn {S['trend_spin']:+.0f} % per 10 years (p = {p_sp:.2f})")
S['bien'] = bien
S['time'] = ts_main.set_index('Location')

# SPIn per year
fig, axes = plt.subplots(2, 1, figsize=(10, 7.2))
ax = axes[0]
x = np.arange(len(YEARS))
wbar = 0.38
for k, loc in enumerate(POLLEN_SITES):
    v = ind_main[ind_main.location == loc].set_index('year')['SPIn'].reindex(YEARS)
    ax.bar(x + (k - 0.5) * wbar, v.values, width=wbar - 0.04, color=COLORS[loc], label=loc)
ax.set_xticks(x)
ax.set_xticklabels(YEARS)
ax.set_ylabel('SPIn (pollen·day m$^{-3}$)')
ax.set_title('a) Seasonal Pollen Integral per year, 2016–2026', loc='left', fontsize=10)
ax.legend(frameon=False, ncol=2)
tidy(ax)
ax = axes[1]
h = helmond_long.set_index('year')['SPIn']
hv = h.reindex(range(h.index.min(), h.index.max() + 1))
ax.bar(hv.index, hv.values, color=COLORS['Helmond'], width=0.8, label='Helmond')
ax.axhline(h.mean(), color='#444444', lw=1, ls='--', label=f'mean {h.index.min()}–{h.index.max()}')
ax.legend(frameon=False, ncol=2)
missing_years = [int(y) for y in hv.index[hv.isna()]]
ax.set_title('b) Helmond, full series' + (f' (no data: {", ".join(map(str, missing_years))})' if missing_years else ''),
             loc='left', fontsize=10)
ax.set_ylabel('SPIn (pollen·day m$^{-3}$)')
tidy(ax)
fig.suptitle('Year-to-year variation in Betula season intensity', x=0.01, ha='left', fontsize=11)
fig.tight_layout()
savefig(fig, 'rq1_spin_per_year.png')
plt.show()

# onset per year, 2016 to 2026
fig, ax = plt.subplots(figsize=(10, 4.2))
for loc in POLLEN_SITES:
    g = ind_main[ind_main.location == loc].sort_values('year')
    d = g.onset.apply(lambda t: pd.Timestamp(2001, t.month, t.day))
    ax.plot(g.year, d, marker=MARKERS[loc], color=COLORS[loc], lw=1.6, ms=7, label=loc)
    mdoy = g.onset_doy.mean()
    m = pd.Timestamp(2001, 1, 1) + pd.Timedelta(days=mdoy - 1)
    lab = pd.Timestamp(2001, 1, 1) + pd.Timedelta(days=round(mdoy) - 1)
    ax.axhline(m, color=COLORS[loc], ls=':', lw=1.2, label=f'{loc} mean ({lab.day} {lab:%b})')
ax.set_xticks(YEARS)
ax.yaxis.set_major_locator(mdates.DayLocator(bymonthday=[1, 4, 8, 12, 16, 20, 24, 28]))
ax.yaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
ax.set_ylabel('Onset date')
ax.set_title('Onset of the Betula pollen season per year, 2016–2026 (5 % of season total)', loc='left', fontsize=11)
tidy(ax)
ax.legend(frameon=False, ncol=4, loc='upper center', bbox_to_anchor=(0.5, -0.1))
fig.tight_layout()
savefig(fig, 'rq1_onset_per_year.png')
plt.show()

## Step 5: Difference between Leiden and Helmond
Compute the difference Leiden minus Helmond per year with a 95 % interval from counting noise (Monte Carlo), and test whether it is systematic with a sign test.

In [ ]:
rng = np.random.default_rng(SEED)

def mc_indicators(s, year, n=N_MC):
    # repeat the calculation n times with noise on the daily counts
    a, b = WINDOW
    s = s[(s.index >= pd.Timestamp(year, *a)) & (s.index <= pd.Timestamp(year, *b))].sort_index().fillna(0.0)
    v, doy = s.values.astype(float), s.index.dayofyear.values
    noise = np.exp(rng.normal(-NOISE_SIGMA ** 2 / 2, NOISE_SIGMA, size=(n, len(v))))   # mean 1
    V = v[None, :] * noise
    C = V.cumsum(axis=1) / V.sum(axis=1, keepdims=True)
    i_on, i_end = (C >= ONSET_PCT).argmax(axis=1), (C >= END_PCT).argmax(axis=1)
    k = np.arange(len(v))[None, :]
    spin = np.where((k >= i_on[:, None]) & (k <= i_end[:, None]), V, 0.0).sum(axis=1)
    return {'onset_doy': doy[i_on], 'end_doy': doy[i_end], 'length': doy[i_end] - doy[i_on] + 1, 'SPIn': spin}

mc = {}
for r in ind_main.itertuples():
    s = pollen[(pollen.location == r.location) & (pollen.year == r.year)].set_index('date')['betula']
    mc[(r.location, r.year)] = mc_indicators(s, r.year)

unc = pd.DataFrame([{'location': loc, 'year': yr, 'onset_sd': np.std(m['onset_doy']), 'end_sd': np.std(m['end_doy']),
                     'spin_cv': 100 * np.std(m['SPIn']) / np.mean(m['SPIn'])} for (loc, yr), m in mc.items()])
S['mc_onset_sd'] = unc.onset_sd.mean()
S['mc_end_sd'] = unc.end_sd.mean()
S['mc_spin_cv'] = unc.spin_cv.mean()
print(f"Counting noise only: onset ± {S['mc_onset_sd']:.1f} d (SD), end ± {S['mc_end_sd']:.1f} d, SPIn ± {S['mc_spin_cv']:.1f} %")

rows = []
for yr in YEARS:
    if ('Helmond', yr) not in mc or ('Leiden', yr) not in mc:
        continue
    H, L = mc[('Helmond', yr)], mc[('Leiden', yr)]
    h = ind_main[(ind_main.location == 'Helmond') & (ind_main.year == yr)].iloc[0]
    l = ind_main[(ind_main.location == 'Leiden') & (ind_main.year == yr)].iloc[0]
    row = {'Year': yr}
    for name, col in [('Onset', 'onset_doy'), ('End', 'end_doy'), ('Length', 'length')]:
        lo, hi = np.percentile(L[col] - H[col], [2.5, 97.5])
        row[f'd{name}'], row[f'd{name}_lo'], row[f'd{name}_hi'] = int(l[col] - h[col]), lo, hi
        row[f'd{name}_meas'] = bool(lo > 0 or hi < 0)
    lo, hi = np.exp(np.percentile(np.log(L['SPIn']) - np.log(H['SPIn']), [2.5, 97.5]))
    row['ratio'], row['ratio_lo'], row['ratio_hi'] = l.SPIn / h.SPIn, lo, hi
    row['ratio_meas'] = bool(lo > 1 or hi < 1)
    rows.append(row)
space = pd.DataFrame(rows)

show = pd.DataFrame({'Year': space.Year})
for name in ['Onset', 'End', 'Length']:
    show[f'Δ{name} L−H (d)'] = space[f'd{name}']
    show[f'Δ{name} 95% (noise)'] = [f'[{a:.0f}, {b:.0f}]' for a, b in zip(space[f'd{name}_lo'], space[f'd{name}_hi'])]
    show[f'Δ{name} measurable'] = space[f'd{name}_meas'].map({True: 'yes', False: 'no'})
show['SPIn ratio L/H'] = space.ratio.round(2)
show['SPIn ratio 95% (noise)'] = [f'[{a:.2f}, {b:.2f}]' for a, b in zip(space.ratio_lo, space.ratio_hi)]
show['SPIn measurable'] = space.ratio_meas.map({True: 'yes', False: 'no'})
results['space_per_year'] = show
display(show)

def summarise(v, meas, ratio=False):
    v = pd.Series(v, dtype=float)
    if ratio:
        lv = np.log(v)
        return {'Mean': round(float(np.exp(lv.mean())), 2), 'SD': round(float(lv.std()), 2), 'Min': round(v.min(), 2),
                'Max': round(v.max(), 2), 'Years L > H': int((v > 1).sum()), 'Years equal': int((v == 1).sum()),
                'Years L < H': int((v < 1).sum()), 'Measurable (95%)': f'{int(sum(meas))}/{len(v)}',
                'Sign test p': round(sign_test(lv), 3)}
    return {'Mean': round(v.mean(), 1), 'SD': round(v.std(), 1), 'Min': int(v.min()), 'Max': int(v.max()),
            'Years L > H': int((v > 0).sum()), 'Years equal': int((v == 0).sum()), 'Years L < H': int((v < 0).sum()),
            'Measurable (95%)': f'{int(sum(meas))}/{len(v)}', 'Sign test p': round(sign_test(v), 3)}

space_sum = pd.DataFrame({
    'Onset, Leiden − Helmond (d)': summarise(space.dOnset, space.dOnset_meas),
    'End, Leiden − Helmond (d)': summarise(space.dEnd, space.dEnd_meas),
    'Length, Leiden − Helmond (d)': summarise(space.dLength, space.dLength_meas),
    'SPIn ratio Leiden / Helmond': summarise(space.ratio, space.ratio_meas, ratio=True)}).T
results['space_summary'] = space_sum.reset_index().rename(columns={'index': 'Indicator'})
display(space_sum)

sd_h = ind_main[ind_main.location == 'Helmond'].onset_doy.std()
sd_l = ind_main[ind_main.location == 'Leiden'].onset_doy.std()
S.update(space=space, space_sum=space_sum, sd_h=sd_h, sd_l=sd_l, sd_d=space.dOnset.std())

# difference per year with 95 % interval from counting noise
fig, axes = plt.subplots(2, 1, figsize=(10, 6.8), sharex=True)
ax = axes[0]
for meas, face in [(True, '#333333'), (False, 'white')]:
    m = space[space.dOnset_meas == meas]
    ax.errorbar(m.Year, m.dOnset, yerr=[m.dOnset - m.dOnset_lo, m.dOnset_hi - m.dOnset], fmt='o', ms=7,
                mfc=face, mec='#333333', ecolor='#888888', elinewidth=1.2, capsize=3,
                label='measurable (95 % excludes 0)' if meas else 'not measurable')
ax.axhline(0, color='#222222', lw=0.8)
ax.set_ylabel('Δ onset, Leiden − Helmond (days)')
ax.set_title('a) Onset: positive = Leiden later', loc='left', fontsize=10)
ax.legend(frameon=False, loc='upper left')
tidy(ax)
ax = axes[1]
for meas, face in [(True, '#333333'), (False, 'white')]:
    m = space[space.ratio_meas == meas]
    ax.errorbar(m.Year, m.ratio, yerr=[m.ratio - m.ratio_lo, m.ratio_hi - m.ratio], fmt='s', ms=7,
                mfc=face, mec='#333333', ecolor='#888888', elinewidth=1.2, capsize=3)
ax.axhline(1, color='#222222', lw=0.8)
ax.set_yscale('log')
ax.set_ylim(min(0.8 * space.ratio_lo.min(), 0.8), max(1.25 * space.ratio_hi.max(), 1.25))
yt = [t for t in [0.125, 0.25, 0.5, 1, 2, 4, 8] if ax.get_ylim()[0] <= t <= ax.get_ylim()[1]]
ax.yaxis.set_major_locator(FixedLocator(yt))
ax.yaxis.set_major_formatter(FixedFormatter([f'{t:g}' for t in yt]))
ax.yaxis.set_minor_formatter(NullFormatter())
ax.set_ylabel('SPIn ratio Leiden / Helmond')
ax.set_title('b) Intensity: below 1 = lower SPIn in Leiden', loc='left', fontsize=10)
ax.set_xticks(YEARS)
tidy(ax)
fig.suptitle('Spatial difference between Leiden and Helmond (~110 km) per year, with 95 % interval from counting noise',
             x=0.01, ha='left', fontsize=11)
fig.tight_layout()
savefig(fig, 'rq1_space_difference.png')
plt.show()

## Step 6: KNMI temperatures and onset per station
Read the KNMI daily data, calibrate one shared threshold T* on Helmond and Leiden, and estimate the onset per KNMI station.

In [ ]:
KNMI_URL = 'https://cdn.knmi.nl/knmi/map/page/klimatologie/gegevens/daggegevens/etmgeg_{}.zip'

def _knmi_head(path, n=40000):
    try:
        if path.lower().endswith('.zip'):
            with zipfile.ZipFile(path) as z:
                names = [x for x in z.namelist() if x.lower().endswith('.txt')]
                return z.open(names[0]).read(n).decode('utf-8', errors='replace') if names else ''
        with open(path, encoding='utf-8', errors='replace') as f:
            return f.read(n)
    except Exception:
        return ''

def knmi_station_of(path):
    # station number of a KNMI daily file, None if it is not a KNMI file
    head = _knmi_head(path)
    if 'METEOROLOGISCH' not in head[:400].upper():
        return None
    m = re.search(r'^\s*(\d{2,3}),\s*(\d{8}),', head, re.M)
    return int(m.group(1)) if m else None

def read_knmi(path):
    if path.lower().endswith('.zip'):
        with zipfile.ZipFile(path) as z:
            txt = z.read([x for x in z.namelist() if x.lower().endswith('.txt')][0]).decode('utf-8', errors='replace')
    else:
        with open(path, encoding='utf-8', errors='replace') as f:
            txt = f.read()
    lines = txt.splitlines()
    hi = next(i for i, l in enumerate(lines) if re.match(r'^\s*#?\s*STN,\s*YYYYMMDD', l))
    cols = [c.strip() for c in lines[hi].lstrip('# ').split(',')]
    body = '\n'.join(l for l in lines[hi + 1:] if l.strip() and not l.lstrip().startswith('#'))
    d = pd.read_csv(io.StringIO(body), names=cols, skipinitialspace=True, low_memory=False)
    date = pd.to_datetime(d['YYYYMMDD'].astype(str), format='%Y%m%d')
    tx, tn = pd.to_numeric(d['TX'], errors='coerce'), pd.to_numeric(d['TN'], errors='coerce')
    return pd.Series(((tx + tn) / 20.0).values, index=date.values).sort_index()   # 0.1 °C -> °C, (TX + TN) / 2

def download_knmi(stn):
    os.makedirs(KNMI_DIR, exist_ok=True)
    dest = os.path.join(KNMI_DIR, f'etmgeg_{stn}.zip')
    try:
        import urllib.request
        with urllib.request.urlopen(KNMI_URL.format(stn), timeout=60) as r, open(dest, 'wb') as f:
            f.write(r.read())
        if knmi_station_of(dest) == stn:
            return dest
        os.remove(dest)
    except Exception as e:
        print(f'Station {stn}: download failed ({e.__class__.__name__}), download {KNMI_URL.format(stn)} manually')
    return None

temps = {}
station_files = {}
cands = set()
for p in glob.glob(os.path.join(KNMI_DIR, '*')):
    if os.path.isfile(p) and p.lower().endswith(('.txt', '.zip')):
        cands.add(os.path.normpath(p))
for p in sorted(cands, key=os.path.getmtime, reverse=True):
    stn = knmi_station_of(p)
    if stn is not None and stn not in station_files:
        station_files[stn] = p
missing = [s for s in STATIONS if s not in station_files]
if missing and DOWNLOAD_KNMI:
    for stn in missing:
        p = download_knmi(stn)
        if p:
            station_files[stn] = p
for stn in STATIONS:
    if stn in station_files:
        temps[stn] = read_knmi(station_files[stn])

st_tab = pd.DataFrame([{'Station': stn, 'Name': name,
                        'Data from': temps[stn].dropna().index.min().date() if stn in temps else '',
                        'Data until': temps[stn].dropna().index.max().date() if stn in temps else '',
                        **{f'km to {loc}': round(km((la, lo), LOCATIONS[loc]), 1) for loc in LOCATIONS}}
                       for stn, (name, la, lo) in STATIONS.items()])
results['knmi_stations'] = st_tab
display(st_tab)

def agdd(t, year, until=(6, 30)):
    # cumulative GDD from 1 March, gaps up to 3 days are interpolated, the series stops at a longer gap
    if t is None:
        return None
    s = t[(t.index >= pd.Timestamp(year, *D0)) & (t.index <= pd.Timestamp(year, *until))]
    if s.empty or s.index[0] != pd.Timestamp(year, *D0):
        return None
    s = s.asfreq('D').interpolate(limit=3, limit_area='inside')
    if s.isna().any():
        s = s.loc[:s[s.isna()].index[0] - pd.Timedelta(days=1)]
    return (s - T_BASE).clip(lower=0).cumsum() if len(s) else None

def agdd_on(c, day):
    if c is None or pd.isna(day) or day < c.index[0] or day > c.index[-1]:
        return np.nan
    return float(c.loc[:day].iloc[-1])

def first_reach(c, thr):
    if c is None or pd.isna(thr):
        return pd.NaT
    hit = c[c >= thr]
    return hit.index[0] if len(hit) else pd.NaT

_AG = {}
def ag(src, yr):
    if (src, yr) not in _AG:
        _AG[(src, yr)] = agdd(temps.get(src), yr)
    return _AG[(src, yr)]

def mean_dT(src, ref, yr, start=(3, 1), end=(4, 15)):
    # mean temperature difference source minus reference, 1 March to 15 April
    a, b = temps.get(src), temps.get(ref)
    if a is None or b is None:
        return np.nan
    i0, i1 = pd.Timestamp(yr, *start), pd.Timestamp(yr, *end)
    x = (a.loc[i0:i1] - b.loc[i0:i1]).dropna()
    return float(x.mean()) if len(x) >= 30 else np.nan

# shared threshold: mean T* at the regional station on the observed onset
obs_onset = {(r.location, r.year): r.onset for r in ind_main.itertuples()}
tstar_reg = {}
for (loc, yr), d in obs_onset.items():
    v = agdd_on(ag(REGIONAL_STATION[loc], yr), d)
    if pd.notna(v):
        tstar_reg[(loc, yr)] = v
T_POOLED = float(np.mean(list(tstar_reg.values())))
S['T_pooled'] = T_POOLED
print(f'T* at the regional station ({len(tstar_reg)} station-years): {T_POOLED:.1f} °C-days ('
      + ', '.join(f"{loc} {np.mean([v for (l, y), v in tstar_reg.items() if l == loc]):.1f}" for loc in POLLEN_SITES) + ')')

est_rows = []
for stn, (name, la, lo) in STATIONS.items():
    if stn not in temps:
        continue
    for yr in YEARS:
        c = ag(stn, yr)
        d = first_reach(c, T_POOLED)
        est_rows.append({'Station': stn, 'Name': name, 'Year': yr, 'Estimated onset': d,
                         'doy': d.dayofyear if pd.notna(d) else np.nan,
                         'AGDD on 1 Apr': agdd_on(c, pd.Timestamp(yr, 4, 1))})
est = pd.DataFrame(est_rows)
yrs_all_st = [yr for yr, g in est.groupby('Year') if g.doy.notna().sum() == est.Station.nunique()]   # years with all stations
est_c = est[est.Year.isin(yrs_all_st)]
st_sum = (est_c.groupby(['Station', 'Name'])
          .agg(n=('doy', 'size'), mean_doy=('doy', 'mean'), sd=('doy', 'std'), earliest=('doy', 'min'),
               latest=('doy', 'max'), agdd_1apr=('AGDD on 1 Apr', 'mean')).reset_index())
st_sum['Mean estimated onset'] = st_sum.mean_doy.map(fmt_doy)
st_sum['Earliest'] = st_sum.earliest.map(fmt_doy)
st_sum['Latest'] = st_sum.latest.map(fmt_doy)
NAT_COL = f'vs {STATIONS[NATIONAL_STATION][0]} (d)'
st_sum[NAT_COL] = (st_sum.mean_doy - st_sum.loc[st_sum.Station == NATIONAL_STATION, 'mean_doy'].iloc[0]).round(1)
st_sum = st_sum.sort_values('mean_doy').reset_index(drop=True)
spread = est_c.groupby('Year').doy.agg(lambda x: x.max() - x.min())
S['nat_spread_mean'], S['nat_spread_min'], S['nat_spread_max'] = spread.mean(), spread.min(), spread.max()
S['n_stations'], S['nat_years'] = est_c.Station.nunique(), f'{min(yrs_all_st)}–{max(yrs_all_st)}' if yrs_all_st else ''
S['st_sum'] = st_sum
nice = st_sum[['Station', 'Name', 'n', 'Mean estimated onset', 'Earliest', 'Latest']].assign(
    **{'SD (d)': st_sum.sd.round(1), 'AGDD on 1 Apr (°C·d)': st_sum.agdd_1apr.round(0)})
nice[NAT_COL] = st_sum[NAT_COL]
results['national_estimated_onset'] = nice
results['national_estimated_onset_per_year'] = est.assign(**{'Estimated onset': est['Estimated onset'].map(lambda d: fmt_date(d, True))})
display(nice)
print(f"Difference between the earliest and latest station per year: mean {spread.mean():.1f} days "
      f"(min {spread.min():.0f}, max {spread.max():.0f})")

# estimated onset per station
fig, ax = plt.subplots(figsize=(9.5, 0.5 * (len(st_sum) + 2) + 1.5))
labels = []
for i, r in st_sum.iterrows():
    x0 = pd.Timestamp(2001, 1, 1) + pd.Timedelta(days=int(r.earliest) - 1)
    x1 = pd.Timestamp(2001, 1, 1) + pd.Timedelta(days=int(r.latest) - 1)
    xm = pd.Timestamp(2001, 1, 1) + pd.Timedelta(days=float(r.mean_doy) - 1)
    ax.plot([x0, x1], [i, i], color='#9a9a9a', lw=2)
    ax.plot(xm, i, 'o', color='#333333', ms=7)
    labels.append(f'{r.Name} ({r.Station})')
for k, loc in enumerate(POLLEN_SITES):
    g = ind_main[ind_main.location == loc]
    i = len(st_sum) + k + 0.4
    x0 = pd.Timestamp(2001, 1, 1) + pd.Timedelta(days=int(g.onset_doy.min()) - 1)
    x1 = pd.Timestamp(2001, 1, 1) + pd.Timedelta(days=int(g.onset_doy.max()) - 1)
    xm = pd.Timestamp(2001, 1, 1) + pd.Timedelta(days=float(g.onset_doy.mean()) - 1)
    ax.plot([x0, x1], [i, i], color=COLORS[loc], lw=2, alpha=0.5)
    ax.plot(xm, i, MARKERS[loc], color=COLORS[loc], ms=8)
    labels.append(f'{loc}: observed (pollen)')
ax.set_yticks(list(range(len(st_sum))) + [len(st_sum) + k + 0.4 for k in range(len(POLLEN_SITES))])
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.xaxis.set_major_locator(mdates.DayLocator(bymonthday=[1, 8, 15, 22]))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
tidy(ax, grid='x')
ax.set_title(f'GDD-estimated onset per KNMI station (dot = mean, line = earliest–latest, {S["nat_years"]})',
             loc='left', fontsize=10.5)
fig.tight_layout()
savefig(fig, 'rq1_national_estimated_onset.png')
plt.show()

## Step 7: Temperature layers
Compare the national (De Bilt) and regional KNMI station per pollen station with a leave-one-year-out threshold, against two benchmarks: the mean onset of the other years and the threshold of the other station.

In [ ]:
def layer_sources(loc):
    rs = REGIONAL_STATION[loc]
    srcs = [('National', f'KNMI {STATIONS[NATIONAL_STATION][0]} ({NATIONAL_STATION})', NATIONAL_STATION,
             km(STATIONS[NATIONAL_STATION][1:], LOCATIONS[loc])),
            ('Regional', f'KNMI {STATIONS[rs][0]} ({rs})', rs, km(STATIONS[rs][1:], LOCATIONS[loc]))]
    return [s for s in srcs if s[2] in temps]

# leave-one-year-out error at the regional station over all years
for loc in POLLEN_SITES:
    yrs_l = [yr for yr in YEARS if (loc, yr) in tstar_reg]
    err = []
    for yr in yrs_l:
        thr = np.mean([tstar_reg[(loc, y)] for y in yrs_l if y != yr])
        p = first_reach(ag(REGIONAL_STATION[loc], yr), thr)
        err.append(abs((p - obs_onset[(loc, yr)]).days) if pd.notna(p) else np.nan)
    S[f'mae_reg_all_{loc}'] = np.nanmean(err)

layer_rows, layer_err = [], []
for loc in POLLEN_SITES:
    srcs = layer_sources(loc)
    yrs_l = [yr for yr in YEARS if (loc, yr) in obs_onset]
    ts = {s[2]: {yr: agdd_on(ag(s[2], yr), obs_onset[(loc, yr)]) for yr in yrs_l} for s in srcs}
    n_ok = {s[2]: sum(pd.notna(v) for v in ts[s[2]].values()) for s in srcs}
    srcs = [s for s in srcs if n_ok[s[2]] >= 4]
    common = [yr for yr in yrs_l if all(pd.notna(ts[s[2]][yr]) for s in srcs) and (loc, yr) in tstar_reg]
    if len(common) < 3:
        continue
    yr_txt = f'{min(common)}–{max(common)} (n={len(common)})'
    for layer, label, src, dist in srcs:
        errs, shifts, dts = [], [], []
        for yr in common:
            thr = np.mean([ts[src][y] for y in common if y != yr])   # leave-one-year-out threshold
            p = first_reach(ag(src, yr), thr)
            errs.append((p - obs_onset[(loc, yr)]).days if pd.notna(p) else np.nan)
            p2 = first_reach(ag(src, yr), tstar_reg[(loc, yr)])       # threshold of the regional station
            shifts.append((p2 - obs_onset[(loc, yr)]).days if pd.notna(p2) else np.nan)
            dts.append(mean_dT(src, REGIONAL_STATION[loc], yr))
            layer_err.append({'Location': loc, 'Layer': layer, 'Source': label, 'Year': yr,
                              'LOYO error (d)': errs[-1], 'Shift with regional T* (d)': shifts[-1], 'ΔT (°C)': dts[-1]})
        e, sh = pd.Series(errs, dtype=float), pd.Series(shifts, dtype=float)
        layer_rows.append({'Location': loc, 'Layer': layer, 'Temperature source': label, 'km to trap': round(dist, 1),
                           'Years': yr_txt, 'ΔT vs regional station (°C)': round(np.nanmean(dts), 2),
                           'Shift with regional T* (d)': round(sh.mean(), 1),
                           'Shift range (d)': f'{sh.min():.0f} to {sh.max():.0f}',
                           'LOYO MAE (d)': round(e.abs().mean(), 2), 'LOYO bias (d)': round(e.mean(), 2),
                           'Years within ±3 d': f'{int((e.abs() <= 3).sum())}/{int(e.notna().sum())}'})
    # benchmark 1: mean onset of the other years, no temperature
    doy = {yr: obs_onset[(loc, yr)].dayofyear for yr in common}
    e = pd.Series([np.mean([doy[y] for y in common if y != yr]) - doy[yr] for yr in common])
    layer_rows.append({'Location': loc, 'Layer': 'Reference', 'Temperature source': 'Climatology (mean onset day, no temperature)',
                       'Years': yr_txt, 'LOYO MAE (d)': round(e.abs().mean(), 2), 'LOYO bias (d)': round(e.mean(), 2),
                       'Years within ±3 d': f'{int((e.abs() <= 3).sum())}/{len(e)}'})
    # benchmark 2: threshold of the other pollen station
    other = [l for l in POLLEN_SITES if l != loc][0]
    thr_o = np.mean([v for (l, y), v in tstar_reg.items() if l == other])
    e = pd.Series([(first_reach(ag(REGIONAL_STATION[loc], yr), thr_o) - obs_onset[(loc, yr)]).days for yr in common], dtype=float)
    layer_rows.append({'Location': loc, 'Layer': 'Reference',
                       'Temperature source': f'Regional station, T* calibrated in {other}',
                       'Years': yr_txt, 'LOYO MAE (d)': round(e.abs().mean(), 2), 'LOYO bias (d)': round(e.mean(), 2),
                       'Years within ±3 d': f'{int((e.abs() <= 3).sum())}/{int(e.notna().sum())}'})

layers = pd.DataFrame(layer_rows)
layers_year = pd.DataFrame(layer_err)
results['scale_layers'] = layers
results['scale_layers_per_year'] = layers_year
S['layers'] = layers
display(layers)

# leave-one-year-out error per year and layer
locs = [l for l in POLLEN_SITES if l in layers_year.Location.values]
fig, axes = plt.subplots(len(locs), 1, figsize=(12, 3.4 * len(locs)), sharex=True, squeeze=False)
for ax, loc in zip(axes[:, 0], locs):
    g = layers_year[layers_year.Location == loc]
    srcs = list(dict.fromkeys(g.Source))
    n = len(srcs)
    for j, src in enumerate(srcs):
        gg = g[g.Source == src]
        col, mk = LAYER_STYLE[gg.Layer.iloc[0]]
        ax.plot(gg.Year + (j - (n - 1) / 2) * 0.12, gg['LOYO error (d)'], ls='', marker=mk, ms=7,
                mfc=col, mec=col, mew=1.5, label=src)
    ax.axhline(0, color='#222222', lw=0.8)
    ax.axhspan(-3, 3, color='#efefef', zorder=0)
    ax.set_ylabel('Predicted − observed\nonset (days)')
    ax.set_title(loc, loc='left', fontsize=11, fontweight='bold')
    ax.legend(frameon=False, fontsize=8.5, loc='upper left', bbox_to_anchor=(1.01, 1.0))
    tidy(ax)
axes[-1, 0].set_xticks(YEARS)
fig.suptitle('Leave-one-year-out onset error per temperature layer (grey band = ±3 days)', x=0.01, ha='left', fontsize=11)
fig.tight_layout()
savefig(fig, 'rq1_scale_layers.png')
plt.show()

## Step 8: City sensors in Enschede
Compute the day on which each city sensor reaches the shared threshold and compare the sensors with the network median and with KNMI Twenthe.

In [ ]:
SENSOR_MIN_READINGS = 100   # minimum number of 10-minute values per day (max 144)
SENSOR_MAX_DEV = 2.0        # °C, a sensor-year that deviates more from the network median in March to April is excluded
SENSOR_EXCLUDE = []         # manual exclusions, e.g. [('EnvMon11', 2026)]
SENSORS = {                 # id: (name, lat, lon)
    'EnvMon01': ('VanHeekplein', 52.217816, 6.897820), 'EnvMon02': ('Stationsplein', 52.222348, 6.890876),
    'EnvMon03': ('Volkspark', 52.219360, 6.878553), 'EnvMon04': ('Breugelmansgaarde', 52.222920, 6.898106),
    'EnvMon05': ('Bothoven-Tusveldburg', 52.223321, 6.906857), 'EnvMon06': ('Oude Markt', 52.220234, 6.895434),
    'EnvMon07': ('Noordendorplaan', 52.226307, 6.888409), 'EnvMon08': ('Toekomststraat', 52.228013, 6.865774),
    'EnvMon09': ('Maanstraat', 52.229031, 6.858184), 'EnvMon11': ('Saxion Klimaatplein', 52.220423, 6.886754)}

sens = pd.read_csv(SENSOR_CSV, parse_dates=['day'])   # columns location, day, min, max, mean, count
sens = sens[sens['count'] >= SENSOR_MIN_READINGS].copy()
sens['tmm'] = (sens['min'] + sens['max']) / 2
sens['year'] = sens['day'].dt.year
rows, excl = [], []
for yr, g in sens.groupby('year'):
    W = g.pivot(index='day', columns='location', values='tmm').sort_index()
    W = W[(W.index >= pd.Timestamp(yr, *D0)) & (W.index <= pd.Timestamp(yr, 4, 30))]
    if W.empty or W.index[0] != pd.Timestamp(yr, *D0):
        print(f'{yr}: no data from 1 March, year skipped')
        continue
    W = W.asfreq('D')
    W = W.loc[:, W.notna().sum() >= 55]   # nearly complete March and April
    if W.shape[1] < 3:
        print(f'{yr}: fewer than 3 complete sensors, year skipped')
        continue
    dev = W.mean() - W.mean().median()
    for sid in W.columns:
        if abs(dev[sid]) > SENSOR_MAX_DEV or (sid, yr) in SENSOR_EXCLUDE:
            excl.append(f'{sid} {yr} ({dev[sid]:+.1f} °C)')
            W = W.drop(columns=sid)
    W = W.interpolate(limit=3, limit_area='inside')
    G = (W - T_BASE).clip(lower=0).cumsum()
    reach = {sid: G.index[int(np.argmax(G[sid].values >= T_POOLED))] if (G[sid] >= T_POOLED).any() else pd.NaT
             for sid in G.columns}
    med = pd.Series(reach).dropna().map(lambda d: d.dayofyear).median()
    dev = W.mean() - W.mean().median()
    for sid in G.columns:
        d = reach[sid]
        rows.append({'Year': yr, 'Sensor': sid, 'Name': SENSORS.get(sid, ('?',))[0],
                     'Mean T Mar–Apr (°C)': round(W[sid].mean(), 2), 'ΔT vs network median (°C)': round(dev[sid], 2),
                     'Date T* reached': fmt_date(d), 'Shift vs network median (d)': (d.dayofyear - med) if pd.notna(d) else np.nan,
                     '_date': d})
if excl:
    print('Excluded:', '; '.join(excl))
sens_res = pd.DataFrame(rows)
sens_dates = sens_res.pop('_date')

results['enschede_sensors'] = sens_res
sens_sum = sens_res.groupby('Year').agg(sensors=('Sensor', 'size'),
                                        spread_days=('Shift vs network median (d)', lambda x: x.max() - x.min()),
                                        spread_T=('ΔT vs network median (°C)', lambda x: x.max() - x.min())).reset_index()
sens_sum.columns = ['Year', 'Sensors', 'Spread onset, max − min (d)', 'Spread mean T, max − min (°C)']
results['enschede_sensors_summary'] = sens_sum
coolest = sens_res.groupby('Name')['ΔT vs network median (°C)'].mean().sort_values()
S['sens'] = {'sum': sens_sum, 'coolest': coolest.index[0], 'coolest_dT': coolest.iloc[0],
             'warmest': coolest.index[-1], 'warmest_dT': coolest.iloc[-1]}
display(sens_res.pivot(index=['Sensor', 'Name'], columns='Year', values='Shift vs network median (d)'))
display(sens_sum)

# comparison with KNMI Twenthe (different measuring setup)
for yr in sorted(sens_res.Year.unique()):
    d_st = first_reach(ag(290, yr), T_POOLED)
    dates = [d for d in sens_dates[sens_res.Year == yr] if pd.notna(d)]
    if pd.notna(d_st) and dates:
        med_doy = np.median([d.dayofyear for d in dates])
        print(f'{yr}: KNMI Twenthe reaches T* on {fmt_date(d_st)}, median of the city sensors '
              f'{fmt_doy(med_doy)} ({med_doy - d_st.dayofyear:+.0f} d)')

# map of the sensors, colour = shift from the network median
cmap9 = LinearSegmentedColormap.from_list('div', ['#2a78d6', '#e6e6e6', '#eb6834'])
yrs9 = sorted(sens_res.Year.unique())
fig, axes = plt.subplots(1, len(yrs9), figsize=(6.5 * len(yrs9) + 1.2, 4.6), squeeze=False, constrained_layout=True)
vmax9 = max(2, np.nanmax(np.abs(sens_res['Shift vs network median (d)'])))
for ax, yr in zip(axes[0], yrs9):
    g = sens_res[sens_res.Year == yr]
    lat = [SENSORS[s][1] for s in g.Sensor]
    lon = [SENSORS[s][2] for s in g.Sensor]
    sc9 = ax.scatter(lon, lat, c=g['Shift vs network median (d)'], cmap=cmap9, vmin=-vmax9, vmax=vmax9,
                     s=140, edgecolor='#222222', lw=0.8, zorder=3)
    below = {'EnvMon11', 'EnvMon03', 'EnvMon01'}   # labels below the dot to avoid overlap
    for sid, s, la, lo, v in zip(g.Sensor, g.Name, lat, lon, g['Shift vs network median (d)']):
        ax.annotate(f'{s} ({v:+.0f})', (lo, la), xytext=(0, -13) if sid in below else (0, 9), textcoords='offset points',
                    fontsize=7.5, color='#333333', ha='center')
    ax.set_aspect(1 / math.cos(math.radians(52.22)))
    ax.ticklabel_format(useOffset=False, style='plain')
    ax.margins(x=0.12, y=0.25)
    ax.set_title(str(yr), loc='left', fontsize=10, fontweight='bold')
    ax.set_xlabel('Longitude (°E)')
    tidy(ax, grid=None)
axes[0, 0].set_ylabel('Latitude (°N)')
cb = fig.colorbar(sc9, ax=axes[0].tolist(), shrink=0.8, pad=0.02)
cb.set_label('Onset shift vs network median (days)\n(+ = later)')
fig.suptitle('Temperature-driven onset differences between city sensors in Enschede', x=0.01, ha='left', fontsize=11)
savefig(fig, 'rq1_enschede_sensors.png')
plt.show()

## Step 9: Overview per spatial scale and save
Put the onset differences per spatial scale next to the model error and save all tables to one Excel file.

In [ ]:
sc_rows = [{'Scale': 'Counting noise of one trap', 'Distance': '0 km',
            'Onset difference (days)': f"± {S['mc_onset_sd']:.1f} (SD)", 'Basis': 'Monte Carlo on pollen counts'}]
b = S['sens']['sum']
sc_rows.append({'Scale': 'Within the city (Enschede sensors)', 'Distance': '≤ 3 km',
                'Onset difference (days)': ', '.join(f'{int(r.Year)}: {r["Spread onset, max − min (d)"]:.0f}' for _, r in b.iterrows()),
                'Basis': 'measured temperature → GDD'})
sp = S['space']
sc_rows.append({'Scale': 'Helmond vs Leiden', 'Distance': f"{km(LOCATIONS['Helmond'], LOCATIONS['Leiden']):.0f} km",
                'Onset difference (days)': f"{sp.dOnset.mean():+.1f} (mean), range {sp.dOnset.min()}–{sp.dOnset.max()}",
                'Basis': 'measured pollen'})
sc_rows.append({'Scale': f"Between {S['n_stations']} KNMI stations", 'Distance': 'national',
                'Onset difference (days)': f"{S['nat_spread_mean']:.1f} (mean max − min per year)", 'Basis': 'KNMI temperature → GDD'})
sc_rows.append({'Scale': 'Year to year at one site', 'Distance': '0 km',
                'Onset difference (days)': f"SD {S['sd_h']:.1f} (Helmond), {S['sd_l']:.1f} (Leiden)", 'Basis': 'measured pollen'})
sc_rows.append({'Scale': 'Model error (GDD, regional station, LOYO)', 'Distance': '–',
                'Onset difference (days)': f"MAE {S['mae_reg_all_Helmond']:.1f} (Helmond), {S['mae_reg_all_Leiden']:.1f} (Leiden)",
                'Basis': 'leave-one-year-out'})
scale_tab = pd.DataFrame(sc_rows)
results['scale_overview'] = scale_tab
display(scale_tab)

with pd.ExcelWriter(OUT_XLSX) as xw:
    for name, df in results.items():
        if df is None or len(df) == 0:
            continue
        df.to_excel(xw, sheet_name=name[:31], index=isinstance(df.columns, pd.MultiIndex) or df.index.name is not None)